# Lesson 6 exercise — LoRA

**Goal:** fine-tune `SmolLM2-360M-Instruct`, a model almost 3× bigger than Lesson 5's, while training only about 1% of its parameters. Then match Lesson 5's held-out score and save an adapter of about 17 MB.

**How to use this notebook**
1. In Colab: *Runtime → Change runtime type → T4 GPU*.
2. Run cells top to bottom. Wherever you see `...` after a `# TODO`, replace it with your code.
3. Run the **check** cell under each TODO. ✅ means you've got it; ❌ gives you a hint.
4. Want to re-run with different settings? Start again from **section 1**. It reloads a clean model, because the trainer adds the adapters to `model` in place.

Stuck? Look up the error in the course's troubleshooting sheet (`reference/troubleshooting.html`). Still stuck? Ask on the Hugging Face Forums with your code and the full traceback.

## 0 · Setup, task data and metric
The same three cells as Lesson 5: versions, the extraction data, and `score()`.

In [ ]:
%pip install -q -U "transformers>=5.19,<6" "trl>=1.15,<2" "peft>=0.21,<1" "datasets>=5,<6" "accelerate>=1.15,<2"
import torch, transformers
device = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "| torch", torch.__version__, "| device:", device)

def check(passed, ok_msg, hint):
    print(("✅ " + ok_msg) if passed else ("❌ " + hint))

In [ ]:
import json, random

PROMPT = "Extract the person's name, age and city as JSON."
NAMES = ["Sara", "Omar", "Lina", "Youssef", "Mona", "Karim", "Nour", "Emma", "Liam", "Sofia",
         "Mateo", "Aiko", "Kenji", "Priya", "Arjun", "Chloe", "Lucas", "Fatima", "Diego", "Elena",
         "Leo", "Amara", "Kofi", "Hana", "Ines"]          # last 5 are test-only
CITIES = ["Cairo", "Alexandria", "Paris", "Lagos", "Tokyo", "Lima", "Berlin", "Toronto",
          "Mumbai", "Seoul", "Nairobi", "Madrid", "Dubai", "Oslo", "Austin", "Lisbon",
          "Hanoi", "Accra", "Quito", "Perth"]                # last 4 are test-only
TEMPLATES = [
    "Hi, I'm {name} from {city}. I'm {age}.",
    "{name} ({age}) lives in {city}.",
    "My name is {name}, I am {age} years old and I live in {city}.",
    "New signup: {name}, age {age}, based in {city}.",
    "Hello from {city}! {name} here, just turned {age}.",
    "{city}-based {name}, {age}, is joining the team next week.",
    "Contact: {name} | {age} | {city}",
    "Applicant {name} is {age} and currently resides in {city}.",
    "Meet {name}: {age} years young and loving life in {city}.",   # test-only
    "I moved to {city} last year. Name's {name}, age {age}.",      # test-only
]

def make_rows(n, split="train", seed=0):
    """Return n chat examples in TRL's prompt-completion format."""
    rng = random.Random(seed)
    test = split == "test"
    names = NAMES[20:] if test else NAMES[:20]
    cities = CITIES[16:] if test else CITIES[:16]
    templates = TEMPLATES[8:] if test else TEMPLATES[:8]
    rows = []
    for _ in range(n):
        person = {"name": rng.choice(names), "age": rng.randint(18, 80), "city": rng.choice(cities)}
        text = rng.choice(templates).format(**person)
        rows.append({
            "prompt": [{"role": "user", "content": f"{PROMPT}\n\nText: {text}"}],
            "completion": [{"role": "assistant", "content": json.dumps(person)}],
        })
    return rows

In [ ]:
def score(model, tokenizer, rows, show=3):
    """Greedy-generate a reply for each row; report valid-JSON rate and field accuracy."""
    model.eval()
    valid, fields_right = 0, 0
    for i, row in enumerate(rows):
        gold = json.loads(row["completion"][0]["content"])
        inputs = tokenizer.apply_chat_template(row["prompt"], add_generation_prompt=True,
                                               return_tensors="pt", return_dict=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=40, do_sample=False)
        reply = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        try:
            pred = json.loads(reply)
            valid += 1
        except json.JSONDecodeError:
            pred = {}
        if isinstance(pred, dict):
            fields_right += sum(pred.get(k) == v for k, v in gold.items())
        if i < show:
            print(f"  reply: {reply!r}")
    n = len(rows)
    result = {"valid_json": valid / n, "field_accuracy": fields_right / (3 * n)}
    print(result)
    return result

In [ ]:
from datasets import Dataset

train = Dataset.from_list(make_rows(500, "train", seed=0))
test_rows = make_rows(40, "test", seed=1)          # held-out: unseen names, cities and phrasings
print(len(train), "train rows |", len(test_rows), "test rows")

## 1 · Load the bigger model
Same code as Lesson 5, new checkpoint id. We still load in fp32 (`float32`): the T4 has no native bf16, and 360M parameters × 4 bytes is only about 1.4 GB.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32).to(device)

n_total = sum(p.numel() for p in model.parameters())
print(f"{n_total / 1e6:.0f}M parameters")
print("BEFORE fine-tuning:")
before = score(model, tok, test_rows)

## 2 · Describe the adapter
A `LoraConfig` says where the small trainable matrices go and how big they are.

**TODO 1:** Build a `LoraConfig` with rank `r=8`, `lora_alpha=16`, adapters on **all linear layers**, for a **causal LM** task.

In [ ]:
from peft import LoraConfig

# TODO 1
peft_config = ...

In [ ]:
# check 1
check(isinstance(peft_config, LoraConfig) and peft_config.r == 8 and peft_config.lora_alpha == 16,
      "Rank and alpha are set.",
      "Call LoraConfig(...) with the r and lora_alpha values from the TODO.")
check(isinstance(peft_config, LoraConfig) and peft_config.target_modules == "all-linear"
      and peft_config.task_type == "CAUSAL_LM",
      "Targets and task type are set.",
      "target_modules takes one special string that means every linear layer; task_type is the causal-LM name.")

## 3 · Hand the config to `SFTTrainer`
This is Lesson 5's trainer with one new argument. `SFTTrainer` freezes the model, inserts the adapters and trains only them.

**TODO 2:** Set a LoRA-sized learning rate (`2e-4`, ten times TRL's full fine-tuning default of `2e-5`; Lesson 5 used `5e-5`), and pass your config to the trainer.

In [ ]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir="lora-out", num_train_epochs=1, per_device_train_batch_size=8,
    learning_rate=...,          # TODO 2a
    bf16=False,                 # the T4 has no bf16
    max_length=256, logging_steps=10, save_strategy="no", report_to="none",
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train, processing_class=tok,
                     peft_config=...)   # TODO 2b

In [ ]:
# check 2
from peft import PeftModel
check(isinstance(args.learning_rate, float) and 1e-4 <= args.learning_rate <= 3e-4, "LoRA-sized learning rate.",
      "LoRA wants a learning rate about 10x higher than full fine-tuning. See the TODO text.")
check(isinstance(trainer.model, PeftModel), "The trainer wrapped your model with LoRA adapters.",
      "Pass your LoraConfig to SFTTrainer with the peft_config= argument.")

## 4 · How much will actually train?
In Keras you'd look at "Trainable params" in `model.summary()`. In PyTorch, every parameter has a `requires_grad` flag. The optimizer only updates parameters where it is `True`.

**TODO 3:** Count the parameters of `trainer.model` that have `requires_grad=True` (count elements with `.numel()`).

In [ ]:
# TODO 3
n_trainable = ...

In [ ]:
# check 3
check(isinstance(n_trainable, int) and 0 < n_trainable < 0.03 * n_total,
      "That's the adapter. PEFT's trainer.model.print_trainable_parameters() reports the same kind of count.",
      "Go through trainer.model.parameters(). Add up the sizes of only the parameters the optimizer will update.")
if isinstance(n_trainable, int):
    print(f"trainable: {n_trainable:,} of {n_total:,} ({100 * n_trainable / n_total:.2f}%)")

## 5 · Train and score
Watch the loss in the log: it should fall fast, as in Lesson 5.

In [ ]:
import time

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()   # measure this run only
t0 = time.time()
trainer.train()
print(f"train time: {time.time() - t0:.0f} s")
if device == "cuda":
    print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

In [ ]:
print("AFTER LoRA fine-tuning:")
after = score(trainer.model, tok, test_rows)
check(after["valid_json"] >= 0.9, "Held-out valid-JSON rate is 90% or more: LoRA learned the format.",
      "Lower than expected. Check the learning rate in TODO 2, then re-run from section 1.")

## 6 · Save just the adapter
`save_pretrained` on a LoRA model writes only the adapter: `adapter_config.json` plus `adapter_model.safetensors` (and a `README.md` stub). The base model stays on the Hub, untouched.

**TODO 4:** Save the adapter into `smollm2-360m-json-lora`, then measure its `.safetensors` file in MB (`os.path.getsize` returns bytes).

In [ ]:
import os
out_dir = "smollm2-360m-json-lora"

# TODO 4
...
size_mb = ...

In [ ]:
# check 4
path = os.path.join(out_dir, "adapter_model.safetensors")
check(os.path.exists(path), "Adapter file written.",
      "A PEFT model saves itself the same way any Hugging Face model does. Save trainer.model into out_dir.")
check(isinstance(size_mb, float) and 0 < size_mb < 100, "Size measured in MB.",
      "Get the .safetensors file's size in bytes, then convert bytes to MB.")
if os.path.exists(path) and isinstance(size_mb, float):
    print(sorted(os.listdir(out_dir)))
    print(f"adapter: {size_mb:.1f} MB | the full model in fp32 would be about {n_total * 4 / 1e6:,.0f} MB")

## Stretch (optional)
Re-run from **section 1** for each experiment, so you start from a clean model.
- Change `r` to 2 or 32. How do the trainable %, adapter size and score change?
- Replace `"all-linear"` with `["q_proj", "v_proj"]` (attention only, as in the original LoRA paper). Does one epoch still reach the same score?
- Lesson 9 shows how to load this adapter back onto the base model and merge it.